In [2]:
#all the needed imports are here 
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

import matplotlib.pyplot as plt

#Data collection 
""" while doing my research online i found 2 relevant dataset so
i decided to first check the possibility of combining both the datasets as each are missing some key info 
for example the fraudulent_kaggle_dataset has the main isfruad column that the return dataset doesnt have 
but the return dataset has return reason which is a very important feature 
so intially i will try to comapare the data and try to merge them based on customer_ids 
if i find none or very less data then i will stick with the fradulent datset as it lacks only some columns 


 """
 #checking possibility of combining 
 
return_data = pd.read_csv('ecommerce_returns_synthetic_data.csv')
fraud_data = pd.read_csv('Fraudulent_E-Commerce_Transaction_Data_2.csv')
print(fraud_data.head())
print(return_data.head())
# just by looking at the ids we can say that they dont match so we go ahead with the fraud datset 
fraud_data= fraud_data.sort_values(["Customer ID", "Transaction Date"])


                         Transaction ID                           Customer ID  \
0  15d2e414-8735-46fc-9e02-80b472b2580f  d1b87f62-51b2-493b-ad6a-77e0fe13e785   
1  0bfee1a0-6d5e-40da-a446-d04e73b1b177  37de64d5-e901-4a56-9ea0-af0c24c069cf   
2  e588eef4-b754-468e-9d90-d0e0abfc1af0  1bac88d6-4b22-409a-a06b-425119c57225   
3  4de46e52-60c3-49d9-be39-636681009789  2357c76e-9253-4ceb-b44e-ef4b71cb7d4d   
4  074a76de-fe2d-443e-a00c-f044cdb68e21  45071bc5-9588-43ea-8093-023caec8ea1c   

   Transaction Amount     Transaction Date Payment Method Product Category  \
0               58.09  2024-02-20 05:58:41  bank transfer      electronics   
1              389.96  2024-02-25 08:09:45     debit card      electronics   
2              134.19  2024-03-18 03:42:55         PayPal    home & garden   
3              226.17  2024-03-16 20:41:31  bank transfer         clothing   
4              121.53  2024-01-15 05:08:17  bank transfer         clothing   

   Quantity  Customer Age Customer Location 

In [3]:
# rename it as data 
data =  pd.read_csv('Fraudulent_E-Commerce_Transaction_Data_2.csv')
data = data.sort_values(["Customer ID", "Transaction Date"])
location_fraud_rate = data.groupby("Customer Location")["Is Fraudulent"].mean()

data["Loc_Fraud_Rate"] = data["Customer Location"].map(location_fraud_rate)
data["Customer_IP_Count"] = data.groupby("Customer ID")["IP Address"].transform("nunique")
data["IP_Usage_Count"] = data.groupby("IP Address")["Transaction Amount"].transform("count")
data["IP_Fraud_History"] = data.groupby("IP Address")["Is Fraudulent"].transform("mean")
data.head()


,Transaction ID,Customer ID,Transaction Amount,Transaction Date,Payment Method,Product Category,Quantity,Customer Age,Customer Location,Device Used,IP Address,Shipping Address,Billing Address,Is Fraudulent,Account Age Days,Transaction Hour,Loc_Fraud_Rate,Customer_IP_Count,IP_Usage_Count,IP_Fraud_History
216551,f55779a5-e8c7-4330-b4ad-154d9fb3813a,00000b1f-6bbc-45e6-bdf3-c98631fbf683,123.35,2024-02-21 23:14:07,credit card,electronics,3,44,Jamesmouth,desktop,45.91.59.42,"5986 Michael Burg\nLake Toniside, FL 04833","5986 Michael Burg\nLake Toniside, FL 04833",0,25,23,0.058737,1,1,0.0
225360,b85ad5b4-c07f-4780-b947-466786a1e442,00002f8f-08b1-4e56-a8a6-c89e195ee9a1,44.43,2024-02-04 03:07:59,credit card,toys & games,3,40,Lake Albert,tablet,17.151.62.174,"98766 Eric Pine Suite 517\nNew Michelleside, M...","98766 Eric Pine Suite 517\nNew Michelleside, M...",0,106,3,0.083333,1,1,0.0
753815,730e7d1a-2936-448d-b535-4d938d8672aa,0000438e-38a8-46f0-bef1-a38e9bf5c857,201.60,2024-03-08 01:29:52,debit card,health & beauty,3,50,Robertfurt,tablet,156.209.39.214,"39706 Simmons Springs Apt. 705\nZacharyhaven, ...","39706 Simmons Springs Apt. 705\nZacharyhaven, ...",0,114,1,0.054902,1,1,0.0
153844,841b2404-9ea2-4f20-8e68-f4fb129196de,000077fe-2812-442c-a79e-75bad41b5a0c,143.35,2024-01-21 14:51:55,debit card,clothing,3,45,Andrewchester,tablet,132.27.210.201,"2305 Megan Course Apt. 436\nBreannafurt, IN 25241","2305 Megan Course Apt. 436\nBreannafurt, IN 25241",0,219,14,0.053691,1,1,0.0
814216,f548a4c6-f0aa-4ae0-857c-865ebe2f5d4a,00007ea2-570b-459c-ad1d-b6f3b6baf8b5,345.71,2024-02-13 05:46:45,bank transfer,home & garden,2,27,South Samuelstad,mobile,90.237.180.172,"76032 Catherine Island\nBrooksville, KS 86451","76032 Catherine Island\nBrooksville, KS 86451",0,343,5,0.076923,1,1,0.0


In [4]:
# Data preprocessing && feature engineering 
# now we will remove the transaction id column it makes no sense to have it

data = data.drop(columns = ['Transaction ID'])#(done already)
data.isnull().sum()


Customer ID           0
Transaction Amount    0
Transaction Date      0
Payment Method        0
Product Category      0
Quantity              0
Customer Age          0
Customer Location     0
Device Used           0
IP Address            0
Shipping Address      0
Billing Address       0
Is Fraudulent         0
Account Age Days      0
Transaction Hour      0
Loc_Fraud_Rate        0
Customer_IP_Count     0
IP_Usage_Count        0
IP_Fraud_History      0
dtype: int64

In [5]:
# we are happy to see no missing values 
#now lets process date and time 
data["Transaction Date"] = pd.to_datetime(data["Transaction Date"])
data["Year"] = data["Transaction Date"].dt.year
data["Month"] = data["Transaction Date"].dt.month
data["Day"] = data["Transaction Date"].dt.day
data["Weekday"] = data["Transaction Date"].dt.weekday
data["IsWeekend"] = (data["Weekday"] >= 5).astype(int)
data = data.drop(columns = ['Transaction Date'])
data = data.drop(columns =["Day"])
data= data.drop(columns = ['Year'])# all are 2024

# we need month weekday to know when they are happening 

In [6]:
# now if u look at customer id   it makes no sense to pass that to the model 

data["Cust_Past_Fraud"] = (
    data.groupby("Customer ID")["Is Fraudulent"].cumsum() - data["Is Fraudulent"]
)


data["Cust_Fraud_Flag"] = (data["Cust_Past_Fraud"] > 0).astype(int)

data["Cust_Txn_Count"] = data.groupby("Customer ID").cumcount()
data["Cust_Fraud_Rate"] = (
    data["Cust_Past_Fraud"] / data["Cust_Txn_Count"].replace(0, 1)
)


data = data.drop(columns=["Customer ID"])
data.tail()

,Transaction Amount,Payment Method,Product Category,Quantity,Customer Age,Customer Location,Device Used,IP Address,Shipping Address,Billing Address,...,Customer_IP_Count,IP_Usage_Count,IP_Fraud_History,Month,Weekday,IsWeekend,Cust_Past_Fraud,Cust_Fraud_Flag,Cust_Txn_Count,Cust_Fraud_Rate
1191500,90.01,credit card,clothing,3,46,North Tinatown,mobile,4.183.2.56,"18410 Bobby Island\nWest Richard, NM 28954","18410 Bobby Island\nWest Richard, NM 28954",...,1,1,0.0,3,2,0,0,0,0,0.0
396543,129.77,PayPal,home & garden,2,42,East Paula,tablet,203.196.127.153,"3152 Mcconnell Bypass Suite 831\nClarkville, I...","3152 Mcconnell Bypass Suite 831\nClarkville, I...",...,1,1,0.0,2,0,0,0,0,0,0.0
650694,111.63,debit card,home & garden,1,23,Port Susan,desktop,168.114.237.57,"25539 William Avenue\nHelenport, WI 85450","1768 Bailey Port Apt. 175\nWest Kevinport, MD ...",...,1,1,0.0,1,1,0,0,0,0,0.0
317753,152.69,PayPal,health & beauty,1,42,East Danielle,desktop,203.142.24.47,"4212 Tiffany Canyon\nNew Matthewberg, AZ 17117","4212 Tiffany Canyon\nNew Matthewberg, AZ 17117",...,1,1,0.0,1,1,0,0,0,0,0.0
85079,152.13,debit card,toys & games,1,27,North Anthonytown,desktop,194.193.14.124,"87090 Jessica Forest Suite 137\nLindseyville, ...","87090 Jessica Forest Suite 137\nLindseyville, ...",...,1,1,0.0,2,1,0,0,0,0,0.0


In [7]:
past_fraud_sum = data["Cust_Past_Fraud"].sum()
fraud_flag_sum = data["Cust_Fraud_Flag"].sum()
fraud_rate_sum = data["Cust_Fraud_Rate"].sum()

print("Sum of Cust_Past_Fraud:", past_fraud_sum)
print("Sum of Cust_Fraud_Flag:", fraud_flag_sum)
print("Sum of Cust_Fraud_Rate:", fraud_rate_sum)

Sum of Cust_Past_Fraud: 0
Sum of Cust_Fraud_Flag: 0
Sum of Cust_Fraud_Rate: 0.0


In [8]:
# so clearly it makes no sense to have these fature drop them
data = data.drop(columns=["Cust_Past_Fraud", "Cust_Fraud_Flag", "Cust_Fraud_Rate","Cust_Txn_Count"], errors='ignore')


In [9]:
# now acting on shipping and billing address 
mask = data['Shipping Address']!=data['Billing Address']
len(data[mask])

147523

In [10]:
# so they are not always equal 
# so cant remove one keep other 
data["Address_Mismatch"] = (data["Shipping Address"] != data["Billing Address"]).astype(int)
data = data.drop(columns=["Shipping Address", "Billing Address"])
data.head()

,Transaction Amount,Payment Method,Product Category,Quantity,Customer Age,Customer Location,Device Used,IP Address,Is Fraudulent,Account Age Days,Transaction Hour,Loc_Fraud_Rate,Customer_IP_Count,IP_Usage_Count,IP_Fraud_History,Month,Weekday,IsWeekend,Address_Mismatch
216551,123.35,credit card,electronics,3,44,Jamesmouth,desktop,45.91.59.42,0,25,23,0.058737,1,1,0.0,2,2,0,0
225360,44.43,credit card,toys & games,3,40,Lake Albert,tablet,17.151.62.174,0,106,3,0.083333,1,1,0.0,2,6,1,0
753815,201.60,debit card,health & beauty,3,50,Robertfurt,tablet,156.209.39.214,0,114,1,0.054902,1,1,0.0,3,4,0,0
153844,143.35,debit card,clothing,3,45,Andrewchester,tablet,132.27.210.201,0,219,14,0.053691,1,1,0.0,1,6,1,0
814216,345.71,bank transfer,home & garden,2,27,South Samuelstad,mobile,90.237.180.172,0,343,5,0.076923,1,1,0.0,2,1,0,0


In [11]:
# now lets get the categorical data 
# we know they exits in two types ordinal and nominal payment methond product cat and device used are nominal
# so one hot 
data = pd.get_dummies(
    data,
    columns=["Payment Method", "Product Category","Device Used"],
    drop_first=True
)


In [13]:
# now working with location_fraud_rate for customer location no way i do one hot endocoding  on it 

location_fraud_rate = data.groupby("Customer Location")["Is Fraudulent"].mean()

data["Loc_Fraud_Rate"] = data["Customer Location"].map(location_fraud_rate)
data["Customer_IP_Count"] = data.groupby("Customer ID")["IP Address"].transform("nunique")
data["IP_Usage_Count"] = data.groupby("IP Address")["Transaction Amount"].transform("count")
data["IP_Fraud_History"] = data.groupby("IP Address")["Is Fraudulent"].transform("mean")


'data["Loc_Fraud_Rate"] = data["Customer Location"].map(location_fraud_rate)\ndata["Customer_IP_Count"] = data.groupby("Customer ID")["IP Address"].transform("nunique")\ndata["IP_Usage_Count"] = data.groupby("IP Address")["Transaction Amount"].transform("count")\ndata["IP_Fraud_History"] = data.groupby("IP Address")["Is Fraudulent"].transform("mean")'

In [ ]:
data = data.drop(columns =['Customer Location'])
data = data.drop(columns=["IP Address"])


In [ ]:
data.head()

In [ ]:
data['IP_Fraud_History'].value_counts()


In [ ]:
data['IP_Usage_Count'].value_counts()

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ["Transaction Amount", "Quantity", "Customer Age", "Account Age Days", "Transaction Hour"]

scaler = StandardScaler()
data[num_cols] = scaler.fit_transform(data[num_cols])
# we done standardizing the data
# so almost done with the data preprocessing and feature engineering 

In [ ]:
bool_cols = data.select_dtypes(include='bool').columns
data[bool_cols] = data[bool_cols].astype(int)
data.head()

In [ ]:
data.shape

In [ ]:
data['Is Fraudulent'].value_counts()

In [15]:
# now being realistic i cant train all the data on my pc i dont have the computing power so i \
# will choose 3 : 1 ration of nonfraud fraud 
# if i get reasonable accuracy i will use collab to do better also i will build a pipe line to use on the data
fraud = data[data["Is Fraudulent"] == 1]

nonfraud = data[data["Is Fraudulent"] == 0].sample(
    n=200_000,
    random_state=42
)

data_small = pd.concat([fraud, nonfraud]).sample(frac=1).reset_index(drop=True)

data_small.shape

(273838, 25)

In [ ]:
# now not only i am showing mercy on my laptop but also having diversity i shuffled the data 
# now finally data prep is done now lets move to model training 
data.head()

In [ ]:
final_data = data

In [16]:
data.head()

,Transaction Amount,Quantity,Customer Age,Customer Location,IP Address,Is Fraudulent,Account Age Days,Transaction Hour,Loc_Fraud_Rate,Customer_IP_Count,...,Address_Mismatch,Payment Method_bank transfer,Payment Method_credit card,Payment Method_debit card,Product Category_electronics,Product Category_health & beauty,Product Category_home & garden,Product Category_toys & games,Device Used_mobile,Device Used_tablet
216551,123.35,3,44,Jamesmouth,45.91.59.42,0,25,23,0.058737,1,...,0,False,True,False,True,False,False,False,False,False
225360,44.43,3,40,Lake Albert,17.151.62.174,0,106,3,0.083333,1,...,0,False,True,False,False,False,False,True,False,True
753815,201.60,3,50,Robertfurt,156.209.39.214,0,114,1,0.054902,1,...,0,False,False,True,False,True,False,False,False,True
153844,143.35,3,45,Andrewchester,132.27.210.201,0,219,14,0.053691,1,...,0,False,False,True,False,False,False,False,False,True
814216,345.71,2,27,South Samuelstad,90.237.180.172,0,343,5,0.076923,1,...,0,True,False,False,False,False,True,False,True,False
